# Training Loop Example

In this notebook, we implement a training loop for a basic transformer model in pyTorch on the tiny shakespeare data set, as a hommage to nanoGPT from Andrej Karpathy's brilliant [Neural Networks: Zero to Hero](https://www.youtube.com/playlist?list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ) series.

## Imports and Setup

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm

In [2]:
from adam import Adam

In [3]:
file_path = "../data/input.txt"

with open(file_path, "r") as f:
    dataset = list(f.read())

## Dataset Preparation

In [4]:
chars = sorted(set(dataset))
ctoi = {c:i for i, c in enumerate(chars)}
itoc = {ctoi[c]:c for c in ctoi.keys()}

In [5]:
encode = lambda s: [ctoi[c] for c in s]
decode = lambda s: "".join([itoc[i] for i in s])

In [6]:
data = encode(dataset)
data_tensor = torch.tensor(data, dtype=torch.long)
data_size = data_tensor.size(dim=0)

In [7]:
split = round(data_size*0.9)
data_train = data_tensor[:split]
data_val = data_tensor[split:]

In [8]:
len_train = data_train.size(dim=0)
len_val = data_val.size(dim=0)

In [23]:
print(f"length of training set: {len_train}")
print(f"length of validation set: {len_val}")

length of training set: 1003855
length of validation set: 111539


## Model Initialization

In [9]:
class Model(nn.Module):

    def __init__(self, block_size: int, vocab_size: int, d_model: int, nhead: int, num_layers: int):
        super().__init__()
        self.block_size = block_size

        self.token_embedding = nn.Embedding(vocab_size, d_model)

        self.positional_embedding = nn.Embedding(block_size, d_model)
        self.causal_mask = torch.triu(torch.ones(block_size, block_size), diagonal=1).bool()

        self.encoding_layer = nn.TransformerEncoderLayer(d_model, nhead, batch_first=True)
        self.transformer = nn.TransformerEncoder(self.encoding_layer, num_layers, enable_nested_tensor=False)

        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, inputs: torch.Tensor):
        embeddings = self.token_embedding(inputs)
        pos_tensor = torch.tensor(range(self.block_size), dtype=torch.long)
        pos_embeddings = self.positional_embedding(pos_tensor)
        embeddings += pos_embeddings

        transformer_outputs = self.transformer(embeddings, self.causal_mask)
        logits = self.lm_head(transformer_outputs)
        return logits

In [33]:
block_size = 256
vocab_size = len(chars)
d_model = 24
nhead = 4
num_layers = 6

## Training Loop

### Initialization and Setup

In [34]:
model = Model(block_size, vocab_size, d_model, nhead, num_layers)
model.train()

Model(
  (token_embedding): Embedding(65, 24)
  (positional_embedding): Embedding(256, 24)
  (encoding_layer): TransformerEncoderLayer(
    (self_attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=24, out_features=24, bias=True)
    )
    (linear1): Linear(in_features=24, out_features=2048, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (linear2): Linear(in_features=2048, out_features=24, bias=True)
    (norm1): LayerNorm((24,), eps=1e-05, elementwise_affine=True)
    (norm2): LayerNorm((24,), eps=1e-05, elementwise_affine=True)
    (dropout1): Dropout(p=0.1, inplace=False)
    (dropout2): Dropout(p=0.1, inplace=False)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=24, out_features=24, bias=True)
        )
        (linear1): Linear(in_features=24, out_features=2048, bia

In [35]:
params = model.parameters()
optim = Adam(params)

In [36]:
losses = []

In [37]:
num_iters = 10000

### Actual training loop

In [38]:
for _ in tqdm(range(num_iters)):
    sequence_idx = np.random.randint(0, len_train - block_size - 1)
    x = data_train[sequence_idx : sequence_idx + block_size]
    y = data_train[sequence_idx + 1 : sequence_idx + block_size + 1]
    
    logits = model(x)
    loss = F.cross_entropy(logits, y)

    losses.append(loss.item())

    optim.zero_grad()
    loss.backward()
    optim.step()

100%|██████████| 10000/10000 [04:45<00:00, 34.97it/s]


### Validation

In [ ]:
#TODO @dkoe00: implement loss visualization, implement sampling

In [40]:
model.eval()

val_losses = []

eval_iters = 500

with torch.no_grad():
    for _ in tqdm(range(eval_iters)):
        sequence_idx = np.random.randint(0, len_val - block_size - 1)
        x = data_val[sequence_idx : sequence_idx + block_size]
        y = data_val[sequence_idx + 1 : sequence_idx + block_size + 1]
        
        logits = model(x)
        loss = F.cross_entropy(logits, y)

        val_losses.append(loss.item())

val_loss = np.average(val_losses)
print(f"average validation loss with {eval_iters} iterations: {val_loss}")

100%|██████████| 500/500 [00:01<00:00, 326.56it/s]

average validation loss with 500 iterations: 2.4876456089019774
